# 経済データ分析Ⅱ　第9回
# ロジットモデル

第8回では、括弧の中の1次式を標準正規分布の累積分布関数 $\Phi$ に入れて持ち家の確率を表すプロビットを学んだ。今回は $\Phi$ の代わりにロジスティック分布の累積分布関数を使う**ロジットモデル**（以下、ロジット）を学ぶ。標準正規分布とロジスティック分布の累積分布関数を比べ、その違いから、ロジットの係数がプロビットより大きくなることと予測確率がほとんど同じになることを説明する。最後に、既婚女性が働いていたかどうかを調べた実際のデータで、線形確率モデル、プロビット、ロジットの3つを比べる。

## Notebook の使い方

コードセルを上から順に選び、`Shift + Enter` で実行します。途中のセルから実行して「変数が見つからない」というエラー（`NameError`）が出たら、先頭のセルから実行し直してください。

JupyterLite では **Python (Pyodide)**、ローカルの Jupyter では Python カーネルを使います。第2節のデータは Notebook の中で作り、第7節のデータはパッケージから読み込むので、CSV ファイルの準備は要りません。

## 1. 準備

次のセルで今回使うライブラリを読み込む。第8回と同じものである。

In [ ]:
import sys
if sys.platform == 'emscripten':
    import pyodide_js
    await pyodide_js.loadPackage(['numpy', 'pandas', 'matplotlib', 'statsmodels', 'scipy'])
    import piplite
    await piplite.install('japanize-matplotlib-jlite')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite
import statsmodels.formula.api as smf
from scipy import stats

## 2. 第7回のデータとプロビット

第7回と同じ500世帯のデータを作り直す。シードも同じ202611なので、第7回と同じ値になる。持ち家の真の確率は

$$p_i=\Phi(-2.3+0.04\times年齢_i+0.09\times年収_i)$$

で、年収の単位は百万円である。

In [ ]:
rng = np.random.default_rng(202611)
n = 500
age = rng.integers(25, 65, size=n)
income = 7 * rng.lognormal(mean=-0.08, sigma=0.4, size=n)
p = stats.norm.cdf(-2.3 + 0.04 * age + 0.09 * income)
owned = (rng.random(n) < p).astype(int)
df = pd.DataFrame({'age': age, 'income': income, 'owned': owned})

第8回のプロビットも推定し直しておく。

In [ ]:
probit = smf.probit('owned ~ age + income', data=df).fit(disp=0)
print(probit.params.round(4))

推定値は第8回の第4節と同じで、定数が −2.4818、年齢が0.0431、年収が0.0874である。

## 3. ロジスティック分布の累積分布関数

第7回の2.2節で説明したように、標準正規分布の累積分布関数 $\Phi(z)$ は、どんな $z$ を入れても0と1の間の値を返す。ロジットでは $\Phi$ の代わりに、**ロジスティック分布**という分布の累積分布関数を使う。この関数を $\Lambda(z)$ と書く。$\Lambda(z)$ は

$$\Lambda(z)=\frac{e^z}{1+e^z}$$

で計算できる。$e$ は約2.718の定数で、$e^z$ は `np.exp(z)` で計算する。$e^z$ はいつも正の値なので分子は分母より小さく、$\Lambda(z)$ も0と1の間に入る。

`stats.logistic.cdf()` は $\Lambda(z)$ を計算する。第7回で使った `stats.norm.cdf()` と同じ働きを、ロジスティック分布について行う。$z=1$ での値を、式と `stats.logistic.cdf()` の両方で計算する。

In [ ]:
print(f'e の1乗：{np.exp(1):.3f}')
print(f'式で計算した Λ(1)：{np.exp(1) / (1 + np.exp(1)):.3f}')
print(f'Λ(1)：{stats.logistic.cdf(1):.3f}')

$e^1=2.718$ なので $\Lambda(1)=2.718/(1+2.718)=0.731$ で、`stats.logistic.cdf(1)` も同じ0.731を返す。$\Phi(z)$ は第7回の2.2節で見た面積なので `stats.norm.cdf()` のような関数で求めるが、$\Lambda(z)$ は式に数を入れるだけで計算できる。

$z$ を −3 から3まで1ずつ変えて、$\Phi(z)$ と $\Lambda(z)$、式で計算した $\Lambda(z)$ を表に並べる。`np.array()` は数の並びを、まとめて計算できる形にする。

In [ ]:
zs = np.array([-3, -2, -1, 0, 1, 2, 3])
pd.DataFrame({'z': zs, 'Φ(z)': stats.norm.cdf(zs), 'Λ(z)': stats.logistic.cdf(zs),
              '式で計算した Λ(z)': np.exp(zs) / (1 + np.exp(zs))}).round(3)

最後の2列は同じ値で、どの $z$ でも式と `stats.logistic.cdf()` が同じ値を返す。

$z=0$ ではどちらの累積分布関数も0.5である。$e^0=1$ なので、$\Lambda(0)=1/(1+1)=0.5$ になる。$z=1$ では $\Phi(1)=0.841$、$\Lambda(1)=0.731$ で、$\Lambda$ のほうが0.5に近い。$z=-1$ でも $\Phi(-1)=0.159$、$\Lambda(-1)=0.269$ で、$\Lambda$ のほうが0.5に近い。$z=3$ まで進んでも、$\Lambda(3)=0.953$ は $\Phi(3)=0.999$ ほど1に近づかない。

2つの累積分布関数を重ねて描く。`np.linspace(-4, 4, 200)` は第7回の2.2節と同じく、−4から4までに200個の点を等しい間隔で並べる。

In [ ]:
z = np.linspace(-4, 4, 200)
fig, ax = plt.subplots()
ax.plot(z, stats.norm.cdf(z), label='プロビット：Φ(z)')
ax.plot(z, stats.logistic.cdf(z), label='ロジット：Λ(z)')
ax.set(xlabel='z', ylabel='確率')
ax.legend(loc='upper left')
plt.show()

どちらもS字の曲線で、どんな $z$ を入れても0と1の間の値を返す。2つの曲線は $z=0$ で交わる。$\Lambda$ の曲線は $\Phi$ よりゆるやかに上がり、0や1にゆっくり近づく。

## 4. ロジット

### 4.1. ロジットの式と潜在変数

ロジットでは持ち家の確率を

$$P(y=1\mid x)=\Lambda(\beta_0+\beta_1\,年齢+\beta_2\,年収)$$

と表す。括弧の中はプロビットと同じ1次式で、それを入れる関数が $\Phi$ から $\Lambda$ に替わる。$\Lambda$ も0と1の間の値を返すので、ロジットの予測確率も0と1の間に入る。ロジットモデルは、**ロジスティック回帰**とも呼ばれる。

第8回の3.2節では、潜在変数 $y_i^*=z_i+\varepsilon_i$ が0より大きい世帯を持ち家とした。$z_i$ は括弧の中の1次式、$\varepsilon_i$ は式に入れていない要因で、$\varepsilon_i$ が標準正規分布に従うとすると持ち家になる確率は $\Phi(z_i)$ になった。$\varepsilon_i$ がロジスティック分布に従うとすると、同じ考え方で確率は $\Lambda(z_i)$ になる。

### 4.2. ロジットの推定

`smf.logit('owned ~ age + income', data=df)` は、`smf.probit` と同じ書き方でロジットの式を作る。係数はプロビットと同じく最尤法で推定する。推定値をプロビットと並べ、ロジットの係数をプロビットの係数で割った比も計算する。

In [ ]:
logit = smf.logit('owned ~ age + income', data=df).fit(disp=0)
coef = pd.DataFrame({'プロビット': probit.params, 'ロジット': logit.params})
coef['比'] = coef['ロジット'] / coef['プロビット']
coef.round(4)

ロジットの推定値は定数が −4.0951、年齢が0.0706、年収が0.1475である。係数の符号はプロビットと同じで、年齢が高いほど、年収が多いほど持ち家の確率が高いことを表す。大きさは違い、比はどれも1.6から1.7の間にある。

## 5. 係数の大きさの違い

4.2節の比はどれも1.6〜1.7だった。ロジットの係数がプロビットの約1.6倍になる理由を、2つの累積分布関数から説明する。

第3節の表と図で見たように、$\Lambda$ の曲線は $\Phi$ よりゆるやかに上がる。$\Phi(1)=0.841$ と同じくらいの確率を $\Lambda$ で表すには、1より大きい $z$ を入れなければならない。$z$ を1.6倍してから $\Lambda$ に入れた値 $\Lambda(1.6z)$ を、$\Phi(z)$ と並べる。

In [ ]:
pd.DataFrame({'z': zs, 'Φ(z)': stats.norm.cdf(zs), 'Λ(z)': stats.logistic.cdf(zs),
              'Λ(1.6z)': stats.logistic.cdf(1.6 * zs)}).round(3)

$z=1$ の行では $\Lambda(1.6z)=0.832$ で、$\Lambda(z)=0.731$ より $\Phi(z)=0.841$ に近い。$z=2$ の行でも、$\Lambda(1.6z)=0.961$ は $\Phi(z)=0.977$ に近い。$z$ を1.6倍して $\Lambda$ に入れると、$\Phi(z)$ に近い値になる。

$\Phi(z)$ と $\Lambda(z)$ に $\Lambda(1.6z)$ を加えた3つの曲線を重ねて描く。`linestyle='--'` は線を破線にする指定である。

In [ ]:
fig, ax = plt.subplots()
ax.plot(z, stats.norm.cdf(z), label='Φ(z)')
ax.plot(z, stats.logistic.cdf(z), color='gray', linestyle='--', label='Λ(z)')
ax.plot(z, stats.logistic.cdf(1.6 * z), label='Λ(1.6z)')
ax.set(xlabel='z', ylabel='確率')
ax.legend(loc='upper left')
plt.show()

$\Lambda(1.6z)$ の曲線は $\Phi(z)$ の曲線とほとんど重なる。違いが見えるのは $z$ が −2 より小さいところと2より大きいところで、そこでは $\Lambda(1.6z)$ のほうがわずかに0.5に近い。

ロジットの1次式の値がプロビットの約1.6倍なら、2つのモデルはほぼ同じ確率を表す。1次式の値を1.6倍にするには、定数と係数をすべて約1.6倍にすればよい。4.2節でロジットの係数がプロビットの1.6〜1.7倍になったのは、このためである。

潜在変数 $y^*$ はデータで観測されず、決まった単位を持たない。そのため係数の大きさは、$\varepsilon$ にどちらの分布を仮定するかで変わる。プロビットとロジットで比べられるのは、係数の符号と、係数から計算した確率である。

## 6. 予測確率の比較

第8回の第5節と同じ2つの世帯（25歳で年収1百万円、64歳で年収20百万円）について、2つのモデルの予測確率を並べる。

In [ ]:
two = pd.DataFrame({'age': [25, 64], 'income': [1, 20]})
pd.DataFrame({'プロビット': probit.predict(two), 'ロジット': logit.predict(two)}).round(3)

25歳で年収1百万円の世帯ではプロビットが0.094、ロジットが0.101、64歳で年収20百万円の世帯では0.978と0.967である。係数の大きさは違っても、予測確率はほとんど同じになる。

データの500世帯でも比べる。`.predict()` は括弧の中を空にすると、データの各世帯の予測確率を返す。横軸にプロビット、縦軸にロジットの予測確率をとって点を描き、2つが等しいところに灰色の線を引く。`ax.plot([0, 1], [0, 1])` は点 (0, 0) と点 (1, 1) を結ぶ直線を引く。

In [ ]:
fig, ax = plt.subplots()
ax.scatter(probit.predict(), logit.predict(), s=10)
ax.plot([0, 1], [0, 1], color='gray')
ax.set(xlabel='プロビットの予測確率', ylabel='ロジットの予測確率')
plt.show()

点は灰色の線のほぼ上に並ぶ。2つの予測確率の差がどれだけ大きくなるかを計算する。`np.abs()` は絶対値を計算する。

In [ ]:
print(f'差の絶対値の最大：{np.abs(logit.predict() - probit.predict()).max():.3f}')

差の絶対値は最大でも0.011である。ロジットとプロビットの予測確率がほとんど同じになることは、多くのデータで見られる。

## 7. 実際のデータの例：既婚女性の就業

### 7.1. データ

第7回からここまでは、真の確率を決めて作った人工データを使ってきた。ここからは実際のデータで3つのモデルを比べる。使うのは、1975年の米国の既婚女性753人について、働いていたかどうかを調べたデータである。参考文献の本が17.1節の例に使っているデータで、`wooldridge` というパッケージに `mroz` という名前で入っている。

ブラウザで実行しているときは、準備のセルで日本語のフォントのパッケージを入れたのと同じ `piplite.install()` で、このパッケージを入れる。`woo.dataWoo('mroz')` はデータを表として読み込む。使う列は次の4つである。

- `inlf`：1975年に働いていた女性は1、働いていなかった女性は0
- `educ`：教育を受けた年数
- `age`：年齢
- `kidslt6`：6歳未満の子どもの数

In [ ]:
if sys.platform == 'emscripten':
    await piplite.install('wooldridge')
import wooldridge as woo

mroz = woo.dataWoo('mroz')
mroz[['inlf', 'educ', 'age', 'kidslt6']].head()

1行目の女性は教育を12年受けた32歳で、6歳未満の子どもが1人いて、1975年に働いていた。

働いていた女性の割合を、753人の全体と、6歳未満の子どもの数ごとに求める。第7回の2.4節で見たように、0と1の列の平均は1の割合になる。`.groupby('kidslt6')` は第7回と同じく子どもの数ごとに行をまとめ、`.size()` は区分ごとの行の数（人数）を数える。

In [ ]:
print(f"働いていた女性の割合：{mroz['inlf'].mean():.3f}")
by_kids = mroz.groupby('kidslt6')['inlf']
pd.DataFrame({'人数': by_kids.size(), '働いていた割合': by_kids.mean()}).round(3)

753人のうち、働いていた女性の割合は0.568である。6歳未満の子どもがいない606人では0.619、1人いる118人では0.390、2人いる26人では0.269で、小さい子どもが多い女性ほど働いていた割合が低い。6歳未満の子どもが3人いる女性は3人だけで、3人とも働いていなかった。

### 7.2. 3つのモデルの推定

働いていたかどうか `inlf` を、教育年数、年齢、6歳未満の子どもの数で説明する。第7回の線形確率モデル、第8回のプロビット、今回のロジットを同じ式で推定する。回帰式の文字は3つのモデルで同じなので、`formula` という名前を付けて使い回す。

In [ ]:
formula = 'inlf ~ educ + age + kidslt6'
lpm_m = smf.ols(formula, data=mroz).fit()
probit_m = smf.probit(formula, data=mroz).fit(disp=0)
logit_m = smf.logit(formula, data=mroz).fit(disp=0)

3つのモデルの係数と、ロジットの係数をプロビットの係数で割った比を表に並べる。

In [ ]:
coef_m = pd.DataFrame({'線形確率モデル': lpm_m.params, 'プロビット': probit_m.params,
                       'ロジット': logit_m.params})
coef_m['ロジットとプロビットの比'] = coef_m['ロジット'] / coef_m['プロビット']
coef_m.round(4)

係数の符号は3つのモデルで同じである。教育年数の係数は正、年齢と6歳未満の子どもの数の係数は負なので、教育年数が長い女性ほど働いている確率が高く、年齢が高い女性と小さい子どもが多い女性ほど低い。

線形確率モデルの係数は、第7回の3.1節で見たように確率の違いをそのまま表す。教育年数と年齢が同じなら、6歳未満の子どもが1人多い女性は働いている確率が0.303低い。

ロジットとプロビットの係数の比は1.63〜1.65で、4.2節の人工データの1.6〜1.7とほぼ同じである。第5節で説明した約1.6倍の関係は、実際のデータでも成り立っている。

### 7.3. 予測確率

3人の女性について、働いている確率を3つのモデルで予測する。1人目は教育を12年受けた35歳で、6歳未満の子どもがいない女性である。2人目は教育年数と年齢が1人目と同じで、6歳未満の子どもが1人いる。3人目は教育を5年受けた30歳で、6歳未満の子どもが3人いる女性にする。

In [ ]:
women = pd.DataFrame({'educ': [12, 12, 5], 'age': [35, 35, 30], 'kidslt6': [0, 1, 3]})
pd.DataFrame({'線形確率モデル': lpm_m.predict(women), 'プロビット': probit_m.predict(women),
              'ロジット': logit_m.predict(women)}).round(3)

1人目の女性の予測確率は、線形確率モデルで0.719、プロビットで0.729、ロジットで0.731である。6歳未満の子どもが1人いる2人目の女性では、0.416、0.399、0.393に下がる。表の値で引き算すると、1人目と2人目の差はプロビットで 0.729 − 0.399 = 0.330、ロジットで 0.731 − 0.393 = 0.338 である。どちらのモデルでも、6歳未満の子どもが1人いると働いている確率が0.33〜0.34低い。

3人目の女性の予測確率は、線形確率モデルでは −0.432 になり、0を下回る。プロビットでは0.004、ロジットでは0.012で、どちらも0と1の間に入る。第7回の3.3節で見た0〜1を外れる予測確率は、実際のデータでも起こる。

データの753人でも確かめる。線形確率モデルの予測確率の最小と最大、ロジットとプロビットの予測確率の差の絶対値の最大を計算する。

In [ ]:
print(f'線形確率モデルの予測確率：{lpm_m.predict().min():.3f}〜{lpm_m.predict().max():.3f}')
print(f'ロジットとプロビットの差の絶対値の最大：{np.abs(logit_m.predict() - probit_m.predict()).max():.3f}')

線形確率モデルの予測確率は −0.171 から0.994で、データの753人の中にも予測確率が0を下回る女性がいる。ロジットとプロビットの予測確率の差は最大でも0.013で、第6節の人工データと同じく、2つのモデルはほとんど同じ確率を返す。

教育年数が12年の女性について年齢を30歳から60歳まで動かし、6歳未満の子どもがいないときと1人いるときの予測確率を図に描く。`kids0` は6歳未満の子どもがいない女性の表、`kids1` は1人いる女性の表で、年齢は30から60までに100点並べる。

In [ ]:
ages = np.linspace(30, 60, 100)
kids0 = pd.DataFrame({'educ': 12, 'age': ages, 'kidslt6': 0})
kids1 = pd.DataFrame({'educ': 12, 'age': ages, 'kidslt6': 1})
kids1.head(3).round(2)

ロジットの予測確率を実線、プロビットの予測確率を破線で描く。`ylim=(0, 1.35)` で縦軸の上を空け、凡例が線に重ならないようにする。

In [ ]:
fig, ax = plt.subplots()
ax.plot(ages, logit_m.predict(kids0), color='C0', label='ロジット：6歳未満の子どもがいない')
ax.plot(ages, probit_m.predict(kids0), color='C0', linestyle='--', label='プロビット：6歳未満の子どもがいない')
ax.plot(ages, logit_m.predict(kids1), color='C1', label='ロジット：6歳未満の子どもが1人')
ax.plot(ages, probit_m.predict(kids1), color='C1', linestyle='--', label='プロビット：6歳未満の子どもが1人')
ax.set(xlabel='年齢', ylabel='働いている確率', ylim=(0, 1.35))
ax.legend(loc='upper left')
plt.show()

実線と破線はほとんど重なり、ロジットとプロビットのどちらで予測しても同じ読み方になる。年齢が高いほど働いている確率は低く、6歳未満の子どもが1人いる女性の線は、どの年齢でも子どもがいない女性の線より下にある。

## 8. 結果の読み方

ロジットは、括弧の中の1次式をロジスティック分布の累積分布関数 $\Lambda$ に入れて持ち家の確率を表す。$\Lambda(z)$ は $e^z/(1+e^z)$ で計算でき、$\Phi(z)$ と同じくS字の曲線で、どんな $z$ を入れても0と1の間の値を返す。$\Lambda$ の曲線は $\Phi$ よりゆるやかに上がり、同じ $z$ を入れると $\Lambda(z)$ のほうが0.5に近い。

$\Lambda(1.6z)$ が $\Phi(z)$ に近いので、ロジットの係数はプロビットの約1.6倍になる。このデータでは比が1.6〜1.7で、係数の符号は2つのモデルで同じだった。予測確率は2つの世帯でも500世帯でもほとんど同じだった。係数の大きさは仮定した分布で変わるので、2つのモデルを比べるときは係数から計算した予測確率を並べる。

1975年の米国の既婚女性753人のデータでも、ロジットの係数はプロビットの1.63〜1.65倍で、2つのモデルの予測確率の差は最大でも0.013だった。係数の符号は線形確率モデルを含む3つのモデルで同じである。教育を12年受けた35歳の女性では、6歳未満の子どもが1人いると、いないときより働いている確率が0.33〜0.34低い。教育を5年受けた30歳で6歳未満の子どもが3人いる女性の予測確率は、線形確率モデルでは −0.432 になり、プロビットとロジットでは0と1の間に入った。

## 参考文献

- [Heiss, F. and Brunner, D. (2024) Using Python for Introductory Econometrics, 2nd edition](https://www.urfie.net/downloads/PDF/UPfIE_web.pdf)、17.1.1節、17.1.2節、17.1.4節
- Wooldridge, J. M. (2001) Econometric Analysis of Cross Section and Panel Data, MIT Press、15.3節

## データの出典

- 第7節のデータは、Python の [`wooldridge` パッケージ](https://pypi.org/project/wooldridge/)の `mroz` である。パッケージの説明は、データの出どころを Mroz, T. A. (1987) The Sensitivity of an Empirical Model of Married Women's Hours of Work to Economic and Statistical Assumptions, Econometrica, 55(4), 765–799 としている。

## 関数の資料

- [NumPy：指数関数（`exp`）](https://numpy.org/doc/stable/reference/generated/numpy.exp.html)
- [NumPy：絶対値（`abs`）](https://numpy.org/doc/stable/reference/generated/numpy.absolute.html)
- [SciPy：ロジスティック分布（`stats.logistic`）](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.logistic.html)
- [statsmodels：式によるロジット（`smf.logit`）](https://www.statsmodels.org/stable/api.html#statsmodels-formula-api)
- [pandas：区分ごとの行の数（`size`）](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.size.html)